# RAG Evaluation Notebook

This notebook evaluates different RAG (Retrieval-Augmented Generation) configurations for a Physics Tutor agent using three key metrics.

In [13]:
# Install dependencies
!pip install groq sentence-transformers -q

In [2]:
import getpass
import os
import re
import numpy as np
from groq import Groq

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ API key: ")
client = Groq()

Enter your GROQ API key: ··········


---
## 1. RAG Evaluation Metrics

We evaluate RAG systems using three metrics, each scored 0-1 (binary pass/fail).

### Metric 1: Context Relevance

**What it measures:** Does the retrieved context contain information relevant to answering the question?

**Scoring:** 1 if retrieved context is relevant, 0 if not.

A RAG system that retrieves irrelevant documents will struggle to generate correct answers.

### Metric 2: Answer Groundedness

**What it measures:** Is the generated answer supported by the retrieved context?

**Scoring:** 1 if the answer is grounded in the context, 0 if it contains hallucinated information.

This detects when the LLM ignores the context and makes things up.

### Metric 3: Answer Correctness

**What it measures:** Is the generated answer factually correct compared to the reference answer?

**Scoring:** 1 if correct, 0 if incorrect.

The ultimate test: did the RAG system produce the right answer?

In [10]:
NUM_QUESTIONS = 5  # Set to 10 for full evaluation

def evaluate_rag_response(question, retrieved_context, generated_answer, reference_answer):
    """Evaluate RAG response on 3 metrics using LLM-as-judge (binary 0/1)"""

    eval_prompt = f"""You are evaluating a RAG system. Be STRICT with scoring (0=Fail, 1=Pass).

Question: {question}
Retrieved Context: {retrieved_context}
Generated Answer: {generated_answer}
Reference Answer: {reference_answer}

Evaluate these 3 metrics:

1. Context Relevance (0 or 1):
   Does the retrieved context contain ALL information needed to fully answer the question?
   Score 0 if the context is missing key facts mentioned in the reference answer.

2. Answer Groundedness (0 or 1):
   Is EVERY claim in the generated answer explicitly supported by the retrieved context?
   Score 0 if the answer includes ANY information not found in the context (even if correct).
   Score 0 if the answer adds details, numbers, or explanations beyond what the context provides.

3. Answer Correctness (0 or 1):
   Does the generated answer match the reference answer's key facts?
   Score 0 if any key information is wrong or missing.

Respond ONLY in this format:
Context Relevance: X
Answer Groundedness: X
Answer Correctness: X"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": eval_prompt}],
        temperature=0
    )

    text = response.choices[0].message.content
    scores = {}
    for metric in ["Context Relevance", "Answer Groundedness", "Answer Correctness"]:
        match = re.search(rf"{metric}\s*:\s*([01])", text, re.IGNORECASE)
        scores[metric] = int(match.group(1)) if match else 0
    return scores

---
## 2. Document Corpus

A small physics knowledge base that our RAG system will retrieve from.

In [4]:
# Documents are intentionally split so some questions need multiple docs
documents = [
    # Doc 0: Forces basics
    "Forces cause objects to accelerate. The SI unit of force is the Newton (N).",

    # Doc 1: Newton's Second Law (needs Doc 0 for full understanding)
    "Newton's Second Law states F = ma, where F is force, m is mass, and a is acceleration.",

    # Doc 2: Mass vs Weight distinction
    "Mass is the amount of matter in an object, measured in kilograms. Weight is the gravitational force on an object, calculated as W = mg where g is gravitational acceleration.",

    # Doc 3: Earth's gravity value
    "On Earth's surface, gravitational acceleration g equals 9.8 m/s². On the Moon, g is about 1.6 m/s².",

    # Doc 4: Free fall basics
    "In free fall, objects accelerate due to gravity. Air resistance can slow falling objects.",

    # Doc 5: Vacuum conditions (needs Doc 4 for full picture)
    "In a vacuum, there is no air resistance. All objects fall at the same rate regardless of mass.",

    # Doc 6: Energy types
    "Energy exists in many forms: kinetic, potential, thermal, chemical, and nuclear.",

    # Doc 7: Kinetic energy formula (needs Doc 6 for context)
    "Kinetic energy is calculated as KE = (1/2)mv², where m is mass and v is velocity.",

    # Doc 8: Energy conservation
    "The law of conservation of energy states that energy cannot be created or destroyed, only transformed.",

    # Doc 9: Momentum definition
    "Momentum equals mass times velocity: p = mv. It is a vector quantity with direction.",

    # Doc 10: Momentum conservation (needs Doc 9)
    "In collisions, total momentum is conserved if no external forces act on the system.",

    # Doc 11: Collision types (needs Doc 9 and 10)
    "In elastic collisions, both momentum and kinetic energy are conserved. In inelastic collisions, only momentum is conserved."
]

print(f"Corpus: {len(documents)} documents")

Corpus: 12 documents


---
## 3. Simple RAG Agent

A minimal RAG implementation using cosine similarity for retrieval.

In [14]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model (downloads ~80MB on first run)
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded!")

def get_embedding(text):
    """Get dense embedding using sentence transformer"""
    return embedder.encode(text, convert_to_numpy=True)

def similarity(emb1, emb2):
    """Cosine similarity between two embeddings"""
    return np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))

def retrieve(query, docs, k=2):
    """Retrieve top-k most similar documents"""
    query_emb = get_embedding(query)
    scores = [(doc, similarity(query_emb, get_embedding(doc))) for doc in docs]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, score in scores[:k]]

# Three separate prompt templates
PROMPT_MINIMAL = """Context: {context}

Question: {question}

Answer:"""

PROMPT_STRUCTURED = """Use the context below to answer the question.

Context:
{context}

Question: {question}

Answer based on the context:"""

PROMPT_GROUNDED = """You are a physics tutor. Answer using ONLY the provided context.
Combine information from all context passages to give a complete answer.
If the context is incomplete, say what's missing.

Context:
{context}

Question: {question}

Answer (use all relevant context):"""

def rag_agent(question, docs, k=2, prompt_template=PROMPT_MINIMAL):
    """RAG agent: retrieve context, then generate answer"""

    # Retrieve
    retrieved = retrieve(question, docs, k=k)
    context = "\n---\n".join(retrieved)

    # Generate
    prompt = prompt_template.format(context=context, question=question)

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    return {
        "context": context,
        "answer": response.choices[0].message.content
    }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


---
## 4. Synthetic Evaluation Dataset

10 physics questions with reference answers and the ground-truth context needed.

In [6]:
# Questions designed to need multiple documents for complete answers
eval_dataset = [
    {
        "question": "What is Newton's Second Law and what unit is force measured in?",
        "reference": "Newton's Second Law states F = ma. Force is measured in Newtons (N).",
        "docs_needed": 2  # Needs Doc 0 + Doc 1
    },
    {
        "question": "How do you calculate the weight of an object on Earth?",
        "reference": "Weight W = mg, where m is mass and g is 9.8 m/s² on Earth.",
        "docs_needed": 2  # Needs Doc 2 + Doc 3
    },
    {
        "question": "Why do all objects fall at the same rate in a vacuum?",
        "reference": "In a vacuum there is no air resistance, so gravity accelerates all objects equally regardless of mass.",
        "docs_needed": 2  # Needs Doc 4 + Doc 5
    },
    {
        "question": "What is kinetic energy and can it be destroyed?",
        "reference": "Kinetic energy is KE = (1/2)mv². Energy cannot be destroyed, only transformed to other forms.",
        "docs_needed": 3  # Needs Doc 6 + Doc 7 + Doc 8
    },
    {
        "question": "What happens to momentum and kinetic energy in an elastic collision?",
        "reference": "In elastic collisions, both momentum (p=mv) and kinetic energy are conserved.",
        "docs_needed": 3  # Needs Doc 9 + Doc 10 + Doc 11
    },
    {
        "question": "What is momentum and is it conserved in collisions?",
        "reference": "Momentum is p = mv. Yes, total momentum is conserved in collisions if no external forces act.",
        "docs_needed": 2  # Needs Doc 9 + Doc 10
    },
    {
        "question": "How does weight differ on Earth vs the Moon?",
        "reference": "Weight = mg. On Earth g=9.8 m/s², on Moon g=1.6 m/s², so objects weigh less on Moon.",
        "docs_needed": 2  # Needs Doc 2 + Doc 3
    },
    {
        "question": "What is the difference between elastic and inelastic collisions?",
        "reference": "In elastic collisions both momentum and KE are conserved. In inelastic, only momentum is conserved.",
        "docs_needed": 2  # Needs Doc 10 + Doc 11
    },
    {
        "question": "What forms can energy take and what is the formula for kinetic energy?",
        "reference": "Energy forms include kinetic, potential, thermal, chemical, nuclear. Kinetic energy KE = (1/2)mv².",
        "docs_needed": 2  # Needs Doc 6 + Doc 7
    },
    {
        "question": "How does air resistance affect falling objects compared to a vacuum?",
        "reference": "Air resistance slows falling objects. In a vacuum with no air resistance, all objects fall at the same rate.",
        "docs_needed": 2  # Needs Doc 4 + Doc 5
    }
]

print(f"Evaluation dataset: {len(eval_dataset)} questions")
print(f"Average docs needed per question: {sum(d['docs_needed'] for d in eval_dataset)/len(eval_dataset):.1f}")

Evaluation dataset: 10 questions
Average docs needed per question: 2.2


---
## 5. RAG Configurations

Three RAG variants with different retrieval and prompting strategies.

In [7]:
rag_configs = {
    "Config A: k=1, minimal": {"k": 1, "prompt": PROMPT_MINIMAL},
    "Config B: k=2, structured": {"k": 2, "prompt": PROMPT_STRUCTURED},
    "Config C: k=3, grounded": {"k": 3, "prompt": PROMPT_GROUNDED}
}

print("RAG Configurations:")
for name, config in rag_configs.items():
    print(f"  {name} (retrieves {config['k']} doc(s))")

RAG Configurations:
  Config A: k=1, minimal (retrieves 1 doc(s))
  Config B: k=2, structured (retrieves 2 doc(s))
  Config C: k=3, grounded (retrieves 3 doc(s))


---
## 6. Run Evaluation

In [15]:
results = {name: {"Context Relevance": [], "Answer Groundedness": [], "Answer Correctness": []}
           for name in rag_configs}

test_data = eval_dataset[:NUM_QUESTIONS]  # Limit for testing

for config_name, config in rag_configs.items():
    print(f"\nEvaluating: {config_name}")

    for i, item in enumerate(test_data):
        # Run RAG
        rag_output = rag_agent(
            item["question"],
            documents,
            k=config["k"],
            prompt_template=config["prompt"]
        )

        # Evaluate
        scores = evaluate_rag_response(
            item["question"],
            rag_output["context"],
            rag_output["answer"],
            item["reference"]
        )

        for metric, score in scores.items():
            results[config_name][metric].append(score)

        print(f"  Q{i+1}: {scores}")

print(f"\n(Evaluated {NUM_QUESTIONS} questions per config)")


Evaluating: Config A: k=1, minimal
  Q1: {'Context Relevance': 1, 'Answer Groundedness': 0, 'Answer Correctness': 1}
  Q2: {'Context Relevance': 1, 'Answer Groundedness': 0, 'Answer Correctness': 1}
  Q3: {'Context Relevance': 0, 'Answer Groundedness': 0, 'Answer Correctness': 1}
  Q4: {'Context Relevance': 0, 'Answer Groundedness': 0, 'Answer Correctness': 1}
  Q5: {'Context Relevance': 1, 'Answer Groundedness': 1, 'Answer Correctness': 0}

Evaluating: Config B: k=2, structured
  Q1: {'Context Relevance': 1, 'Answer Groundedness': 1, 'Answer Correctness': 1}
  Q2: {'Context Relevance': 1, 'Answer Groundedness': 1, 'Answer Correctness': 1}
  Q3: {'Context Relevance': 0, 'Answer Groundedness': 0, 'Answer Correctness': 0}
  Q4: {'Context Relevance': 0, 'Answer Groundedness': 0, 'Answer Correctness': 0}
  Q5: {'Context Relevance': 1, 'Answer Groundedness': 1, 'Answer Correctness': 1}

Evaluating: Config C: k=3, grounded
  Q1: {'Context Relevance': 1, 'Answer Groundedness': 1, 'Answer Cor

---
## 7. Results Comparison

In [16]:
print("\n" + "="*70)
print("RAG EVALUATION RESULTS")
print("="*70)
print(f"{'Configuration':<35} {'Relevance':>10} {'Grounded':>10} {'Correct':>10} {'Overall':>10}")
print("-"*70)

best_config = None
best_score = 0

for config_name in rag_configs:
    n = len(results[config_name]["Context Relevance"])
    relevance = sum(results[config_name]["Context Relevance"]) / n
    grounded = sum(results[config_name]["Answer Groundedness"]) / n
    correct = sum(results[config_name]["Answer Correctness"]) / n
    overall = (relevance + grounded + correct) / 3

    print(f"{config_name:<35} {relevance:>10.0%} {grounded:>10.0%} {correct:>10.0%} {overall:>10.0%}")

    if overall > best_score:
        best_score = overall
        best_config = config_name

print("-"*70)
print(f"Best: {best_config} ({best_score:.0%})")


RAG EVALUATION RESULTS
Configuration                        Relevance   Grounded    Correct    Overall
----------------------------------------------------------------------
Config A: k=1, minimal                     60%        20%        80%        53%
Config B: k=2, structured                  60%        60%        60%        60%
Config C: k=3, grounded                   100%       100%       100%       100%
----------------------------------------------------------------------
Best: Config C: k=3, grounded (100%)


---
## 8. Summary

### Key Observations

**Context Relevance**: With k=1, the system often retrieves only partial information. Questions that need 2-3 documents will have incomplete context.

**Answer Groundedness**: The grounded prompt explicitly instructs the model to use context and admit gaps, reducing hallucination.

**Answer Correctness**: Higher k retrieves more complete information, and better prompts synthesize it correctly.

### Why Config C Should Win

| Config | k | What happens |
|--------|---|--------------|
| A (k=1) | 1 | Misses second/third doc needed for complete answers |
| B (k=2) | 2 | Gets most info, but prompt doesn't emphasize combining |
| C (k=3) | 3 | Gets all needed docs + prompt instructs to combine them |

### Takeaway

RAG performance depends on **both** retrieval (getting enough relevant docs) **and** generation (synthesizing information correctly). Config C wins because it retrieves enough context AND uses a prompt that instructs the model to combine information from multiple passages.